In [12]:
import sqlite3
import pandas as pd

# tworzy bazę w pamięci — znika po zamknięciu, idealne do ćwiczeń
conn = sqlite3.connect(':memory:')

# tworzysz tabelę z pandas DataFrame
df_sales = pd.DataFrame({
    'seller': ['Anna', 'Anna', 'Anna', 'Bartek', 'Bartek', 'Bartek'],
    'month': ['2024-01', '2024-02', '2024-03', '2024-01', '2024-02', '2024-03'],
    'sales': [100, 120, 90, 200, 180, 220]
})

# ładujesz DataFrame do bazy
df_sales.to_sql('sales', conn, index=False, if_exists='replace')

# piszesz SQL i dostajesz wynik jako DataFrame
result = pd.read_sql_query("""
    SELECT seller, SUM(sales) as total
    FROM sales
    GROUP BY seller
    ORDER BY total DESC
""", conn)

print(result)

   seller  total
0  Bartek    600
1    Anna    310


In [13]:
conn = sqlite3.connect(':memory:')

df_orders = pd.DataFrame({
    'order_id': range(1, 11),
    'customer_id': [1, 1, 2, 2, 3, 3, 1, 2, 3, 1],
    'month': ['2024-01', '2024-01', '2024-01', '2024-02',
              '2024-02', '2024-02', '2024-03', '2024-03', '2024-03', '2024-03'],
    'amount': [500, 300, 1000, 800, 400, 600, 700, 900, 500, 200],
    'status': ['completed', 'cancelled', 'completed', 'completed',
               'cancelled', 'completed', 'completed', 'completed', 'cancelled', 'completed']
})

df_orders.to_sql('orders', conn, index=False, if_exists='replace')

10

In [14]:
result = pd.read_sql_query("""
    select customer_id, sum(amount)
    from orders
    where status = 'completed'
    group by customer_id
    order by sum(amount) desc
""", conn)

print(result)

   customer_id  sum(amount)
0            2         2700
1            1         1400
2            3          600


In [15]:
result = pd.read_sql_query("""
select customer_id, count(*)
from orders
where status = 'completed'
group by customer_id
having count(*) > 2
""", conn)

print(result)

   customer_id  count(*)
0            1         3
1            2         3


In [16]:
result = pd.read_sql_query("""
select month, sum(amount) as total_amount
from orders
where status = 'completed'
group by month
order by total_amount desc
LIMIT 1
""", conn)

print(result)

     month  total_amount
0  2024-03          1800


In [23]:
result = pd.read_sql_query("""
with ranked_data as(
    select month, sum(amount) as total_amount,
    row_number() over(order by sum(amount) desc) as rn
    from orders
    where status = 'completed'
    group by month
)
    select month, total_amount
    from ranked_data
    where rn = 1
""", conn)

print(result)

     month  total_amount
0  2024-03          1800


In [24]:
df_sales = pd.DataFrame({
    'seller_id': [1, 1, 1, 2, 2, 2, 3, 3, 3],
    'month': ['2024-01', '2024-02', '2024-03',
              '2024-01', '2024-02', '2024-03',
              '2024-01', '2024-02', '2024-03'],
    'revenue': [10000, 12000, 9000, 15000, 14000, 16000, 8000, 9000, 11000],
    'region': ['North', 'North', 'North', 'South', 'South', 'South', 'North', 'North', 'North']
})

df_sales.to_sql('sales', conn, index=False, if_exists='replace')

9

In [41]:
result = pd.read_sql_query("""
select *, lag(revenue, 1, 0) over (partition by seller_id order by month) as revenue_prev,
rank() over (partition by month order by revenue desc) as rank_in_month,
SUM(revenue) OVER (PARTITION BY seller_id ORDER BY month) AS running_total
from sales
""", conn)

print(result)

   seller_id    month  revenue region  revenue_prev  rank_in_month  \
0          1  2024-01    10000  North             0              2   
1          1  2024-02    12000  North         10000              2   
2          1  2024-03     9000  North         12000              3   
3          2  2024-01    15000  South             0              1   
4          2  2024-02    14000  South         15000              1   
5          2  2024-03    16000  South         14000              1   
6          3  2024-01     8000  North             0              3   
7          3  2024-02     9000  North          8000              3   
8          3  2024-03    11000  North          9000              2   

   running_total  
0          10000  
1          22000  
2          31000  
3          15000  
4          29000  
5          45000  
6           8000  
7          17000  
8          28000  


In [42]:
df_employees = pd.DataFrame({
    'employee_id': range(1, 8),
    'name': ['Anna', 'Bartek', 'Celina', 'Darek', 'Ewa', 'Filip', 'Gosia'],
    'department': ['IT', 'IT', 'HR', 'HR', 'IT', 'HR', 'IT'],
    'salary': [8000, 9500, 6000, 7000, 8500, 6500, 10000],
    'years_exp': [3, 5, 2, 4, 4, 3, 7]
})

df_employees.to_sql('employees', conn, index=False, if_exists='replace')

7

In [48]:
result = pd.read_sql_query("""
with avg_salary as (
select department, avg(salary) as avg_salary
from employees
group by department
)
select e.name, e.department, e.salary, a.avg_salary, e.salary - a.avg_salary as salary_diff,
rank() over (partition by e.department order by salary desc) as rank_in_department
from employees e
join avg_salary a on e.department = a.department
""", conn)

print(result)

     name department  salary  avg_salary  salary_diff  rank
0   Darek         HR    7000      6500.0        500.0     1
1   Filip         HR    6500      6500.0          0.0     2
2  Celina         HR    6000      6500.0       -500.0     3
3   Gosia         IT   10000      9000.0       1000.0     1
4  Bartek         IT    9500      9000.0        500.0     2
5     Ewa         IT    8500      9000.0       -500.0     3
6    Anna         IT    8000      9000.0      -1000.0     4
